# Insecticide-resistance alleles in African malaria mosquitoes
### Where else are the newly reported *Vgsc* alleles V402L and I1527T found?

**Background.** Pyrethroid insecticides (used on bed nets) kill mosquitoes by jamming the voltage-gated sodium
channel, encoded by the gene *Vgsc*. Amino-acid changes in this channel make mosquitoes resistant ("knock-down
resistance", *kdr*). The established resistance alleles in *Anopheles gambiae* and *An. coluzzii* are **L995F**,
**L995S** and **N1570Y**. A longitudinal study in Burkina Faso (Kientega et al. 2024, *Malaria Journal* 23:280)
reported two newer alleles, **V402L** and **I1527T**, rising in frequency there.

**Question.** Across all the mosquitoes in the MalariaGEN *Anopheles gambiae* 1000 Genomes resource (Ag3), where else
do V402L and I1527T occur, how common are they, and do they occur together with the classic *kdr* alleles?

**Answer key.** The classic alleles have well-documented geographic patterns (e.g. Clarkson et al. 2021,
*Molecular Ecology*). If this notebook reproduces those, the method is working.

**How to run.** In Colab: `Runtime → Run all`, or run cells one at a time with Shift+Enter. When asked, allow
access to your Google account (this is how MalariaGEN's cloud data is read). The slowest cells take a few minutes.
At the end, a file `results.zip` downloads to your computer.

*Data: MalariaGEN Ag3. Read the terms of use (step 2) before publishing any result.*

## Step 1. Install the software and connect to the data
`malariagen_data` is MalariaGEN's Python package. It reads genotype data straight from Google Cloud, so nothing
large is downloaded to your computer.

**Before running:** your Google account must be registered with MalariaGEN (data access request form on the
Vector Observatory data-access page). When Colab asks for access, choose **Select all**.

**Check the output of the next cell:** "Client location" should say *United States*. If it does not, use
`Runtime → Disconnect and delete runtime`, reconnect, and run again (MalariaGEN asks users to work from US-based
Colab machines, where the data are stored).

Only sample sets marked for **unrestricted use** are loaded, so the results can be shared publicly.

In [ ]:
%pip install -q --no-warn-conflicts malariagen_data

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import malariagen_data

os.makedirs("results", exist_ok=True)
# unrestricted_use_only=True: use only sample sets with no embargo on publishing results,
# so everything this notebook produces can go on GitHub.
ag3 = malariagen_data.Ag3(unrestricted_use_only=True)
print("malariagen_data version:", malariagen_data.__version__)
ag3

## Step 2. What samples are available?
A **sample set** is one batch of mosquitoes contributed by one study. Each mosquito has metadata: where and when it
was collected, and which species (`taxon`) it is.

Because the client was created with `unrestricted_use_only=True`, only sample sets without a publication embargo
are listed. The `terms_of_use` columns are still worth reading.

In [ ]:
df_sets = ag3.sample_sets()
df_sets.to_csv("results/sample_sets.csv", index=False)
print(len(df_sets), "sample sets")
df_sets.head(10)

In [ ]:
df_samples = ag3.sample_metadata()
print(df_samples.shape[0], "mosquitoes in total")
df_counts = ag3.count_samples(index="country", columns="taxon")
df_counts.to_csv("results/sample_counts_by_country.csv")
df_counts

## Step 3. Amino-acid allele frequencies in *Vgsc*
`AGAP004707-RD` is the reference transcript of *Vgsc*; all the allele names used here (L995F, V402L, ...) are
numbered on this transcript.

Mosquitoes are grouped into **cohorts**: same region (admin level 1), same year and same species. Cohorts with fewer than
10 mosquitoes are dropped, because frequencies from tiny groups are unreliable.

This cell is the slowest (a few minutes): it reads the genotypes of every mosquito across the whole gene.

In [ ]:
TRANSCRIPT = "AGAP004707-RD"   # Vgsc
MIN_COHORT = 10

df_aa = ag3.aa_allele_frequencies(
    transcript=TRANSCRIPT,
    cohorts="admin1_year",
    min_cohort_size=MIN_COHORT,
    drop_invariant=True,
    include_counts=True,
)
df_aa.to_csv("results/vgsc_aa_frequencies_all.csv")
print(df_aa.shape[0], "amino-acid changes found;",
      sum(c.startswith("frq_") for c in df_aa.columns), "cohorts")

## Step 4. Overview: every common amino-acid change, in every cohort
Rows are amino-acid changes found at ≥ 5 % frequency in at least one cohort. Columns are cohorts. Darker red =
more common.

In [ ]:
df_common = df_aa.query("max_af >= 0.05").sort_values("position")
print(len(df_common), "amino-acid changes at >= 5% in at least one cohort")
fig = ag3.plot_frequencies_heatmap(df_common, max_len=None, show=False)
fig.write_html("results/vgsc_heatmap.html")
fig.show()

## Step 5. The five alleles of interest
Turn the wide table (one column per cohort) into a long table (one row per allele per cohort), and look up each
cohort's country, region, species and year in the sample metadata.

In [ ]:
FOCAL = ["V402L", "L995F", "L995S", "I1527T", "N1570Y"]

d = df_aa.reset_index()
d = d[d["aa_change"].isin(FOCAL)]
frq_cols = [c for c in d.columns if c.startswith("frq_")]
cohorts = [c[len("frq_"):] for c in frq_cols]

rows = []
for _, r in d.iterrows():
    for c in cohorts:
        rows.append({"allele": r["aa_change"], "position": r["position"], "cohort": c,
                     "frequency": r[f"frq_{c}"],
                     "alleles_called": r.get(f"nobs_{c}", np.nan),
                     "alt_count": r.get(f"count_{c}", np.nan)})
long = pd.DataFrame(rows)

# Look up country, region, species and year for each cohort from the sample metadata
# (more reliable than parsing the cohort label).
lookup = (df_samples.dropna(subset=["cohort_admin1_year"])
          .groupby("cohort_admin1_year")
          .agg(country=("country", "first"), admin1=("admin1_iso", "first"),
               taxon=("taxon", "first"), year=("year", "first"), n_mosquitoes=("sample_id", "size")))
long = long.merge(lookup, left_on="cohort", right_index=True, how="left")
long["country_code"] = long["admin1"].str.split("-").str[0]
long.to_csv("results/focal_alleles_by_cohort.csv", index=False)
print(len(FOCAL) - long["allele"].nunique(), "of the focal alleles were not found at all")
long.head()

### Check the method against what is already known (the answer key)
Where are the classic *kdr* alleles common? Compare this table with the published picture: L995F common in West
Africa, L995S mainly in Central and East Africa, N1570Y on the L995F background in West Africa.

In [ ]:
present = long[long["frequency"] > 0]
summary = (present.groupby("allele")
           .agg(cohorts_present=("cohort", "nunique"),
                countries=("country", lambda s: ", ".join(sorted(set(s.dropna())))),
                max_frequency=("frequency", "max"))
           .reindex(FOCAL))
summary["cohorts_tested"] = long.groupby("allele")["cohort"].nunique().reindex(FOCAL)
top = present.loc[present.groupby("allele")["frequency"].idxmax(), ["allele", "cohort"]].set_index("allele")
summary["highest_in"] = top["cohort"]
summary.to_csv("results/focal_alleles_summary.csv")
summary

## Step 6. The question: where are V402L and I1527T?

In [ ]:
for allele in ["V402L", "I1527T"]:
    sub = present[present["allele"] == allele].sort_values("frequency", ascending=False)
    print(f"\n{allele}: found in {sub['cohort'].nunique()} cohort(s)")
    print(sub[["cohort", "country", "taxon", "year", "n_mosquitoes", "frequency", "alt_count", "alleles_called"]].to_string(index=False))

### Figure: the five alleles across all cohorts
Each dot is one cohort where the allele was found. Cohorts are ordered by country, then region, then year.

In [ ]:
plot = present.sort_values(["country_code", "admin1", "year"])
order = list(dict.fromkeys(plot["cohort"]))
ypos = {a: i for i, a in enumerate(FOCAL[::-1])}
xpos = {c: i for i, c in enumerate(order)}
colour = {"coluzzii": "#2a78d6", "gambiae": "#eb6834"}

fig, ax = plt.subplots(figsize=(max(8, 0.16 * len(order)), 3.6))
for _, r in plot.iterrows():
    ax.scatter(xpos[r["cohort"]], ypos[r["allele"]], s=20 + 180 * r["frequency"],
               color=colour.get(r["taxon"], "#6b6a66"), alpha=0.8, linewidths=0)
ax.set_yticks(list(ypos.values()), list(ypos.keys()))
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=6)
ax.set_xlim(-1, len(order))
ax.grid(axis="y", color="#e4e3de")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for t, c in [("An. coluzzii", "#2a78d6"), ("An. gambiae", "#eb6834"), ("other / hybrid", "#6b6a66")]:
    ax.scatter([], [], color=c, label=t)
ax.legend(loc="upper right", frameon=False, fontsize=8)
ax.set_title("Vgsc resistance alleles: cohorts where each was found (dot size = frequency)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig("results/focal_alleles_by_cohort.png", dpi=200)
plt.show()

## Step 7. Change over time in Burkina Faso
The 2024 study reported V402L and I1527T increasing in Burkina Faso. This groups Burkina Faso mosquitoes by region and
year and plots each allele's frequency with 95 % confidence intervals.

In [ ]:
ds_bf = ag3.aa_allele_frequencies_advanced(
    transcript=TRANSCRIPT,
    area_by="admin1_iso",
    period_by="year",
    sample_query="country == 'Burkina Faso'",
    min_cohort_size=MIN_COHORT,
    variant_query="aa_change in " + str(FOCAL),
)
df_bf = ds_bf[["variant_label", "event_frequency", "event_frequency_ci_low", "event_frequency_ci_upp",
               "event_count", "event_nobs"]].to_dataframe()
df_bf.to_csv("results/burkina_faso_time_series.csv")
fig = ag3.plot_frequencies_time_series(ds_bf, show=False)
fig.write_html("results/burkina_faso_time_series.html")
fig.show()

## Step 8. Save everything
This zips the `results` folder and downloads it. Send `results.zip` back so the README can be written from the real
numbers.

In [ ]:
import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not in Colab: results.zip is in the current folder.")

## Questions to answer in your own words (interview preparation)
1. What does *Vgsc* encode, and why do changes in it make mosquitoes resistant to pyrethroids?
2. What is a cohort here, and why are cohorts with fewer than 10 mosquitoes dropped?
3. Allele frequency = alt_count / alleles_called. Why is `alleles_called` about twice the number of mosquitoes?
4. Did the classic alleles (L995F, L995S, N1570Y) appear where the published studies say they should?
5. Where else are V402L and I1527T found? Are they in the same cohorts as L995F?
6. Finding an allele in a cohort is not proof it causes resistance. What other evidence would you need?
7. Ag3 samples were not collected evenly across Africa or across years. How does that limit the conclusions?